# Week 2 Day 1 — Agent Foundations
## Reasoning Loops, Tool Calling & Raw Python Agents

**Goal:** Build a minimal agent from scratch using raw Python + the Anthropic Messages API.

**Constraints**
- No LangChain
- No LangGraph
- No CrewAI
- Explicit agent loop
- Explicit JSON tool schemas
- Explicit tool execution
- Explicit conversation state
- Maximum-iteration safeguard
- Logging for tool calls and observations

> **Security:** Never hard-code an Anthropic API key. Set `ANTHROPIC_API_KEY` as an environment variable.

## Learning objectives

By the end of this notebook, the implementation demonstrates:

1. Agent vs. chatbot vs. workflow.
2. The ReAct-style `Reason → Act → Observe → repeat` mental model.
3. Anthropic tool definitions with JSON schemas.
4. A single tool-call request and manual `tool_result` return.
5. A raw Python agent loop.
6. A multi-step task requiring at least two tool calls.
7. Conversation memory vs. working state.
8. Logging and debugging.
9. Deliberate failure testing and guardrails.
10. Why frameworks exist after understanding the primitives.

## Task 1 — Agent Concepts & Mental Model

### Agent vs. chatbot vs. workflow

- **Chatbot:** Primarily responds to a user's message. It may keep conversation history but does not necessarily choose actions or operate external tools.
- **Workflow:** A predefined sequence of steps where the developer decides the control flow.
- **Agent:** An LLM-driven loop that can decide what action to take next, call tools, inspect their results, and continue until it can produce a final answer.

### What makes a system agentic?

- **Autonomy:** the model chooses the next step.
- **Tool use:** it can interact with external capabilities.
- **Multi-step planning:** one request can require several actions.
- **Observation/self-correction:** tool results influence the next step.
- **State:** the system preserves enough context to continue the task.

### ReAct mental model

```text
User request
     |
     v
+-----------+
|  REASON   |  <- model decides what is needed
+-----------+
     |
     v
+-----------+
|    ACT    |  <- choose a tool + arguments
+-----------+
     |
     v
+-----------+
|  OBSERVE  |  <- execute tool and return result
+-----------+
     |
     +------> Complete? -- No --> REASON
                       |
                      Yes
                       v
                 Final answer
```

Pseudocode:

```python
while iterations < max_iterations:
    response = model(messages, tools)

    if response contains tool_use:
        execute selected tool
        append assistant response
        append tool_result
        continue

    return final_text
```

### When is an agent overkill?

An agent is overkill when the task has a known deterministic sequence and no meaningful choice is required. A simple prompt, ordinary Python function, or fixed workflow is usually cheaper, easier to test, and easier to debug. Use autonomy only when the uncertainty or branching justifies it.

## Task 2 — Tool Calling Fundamentals

Two tools are used:

1. `calculator` — evaluates restricted arithmetic.
2. `get_weather` — a deterministic weather lookup **stub** for learning; it is not a live weather service.

### Why tool descriptions matter

The model does not execute Python functions directly. It receives the tool name, description, and JSON schema and uses them to decide whether a tool is appropriate and what arguments to send.

Good descriptions should:
- state exactly what the tool does,
- explain limitations,
- define each input clearly,
- make units/formats explicit,
- avoid vague names.

The schema is a contract between the model and the application.

In [ ]:
import os
import json
import ast
import operator as op
from typing import Any, Dict

try:
    from anthropic import Anthropic
except ImportError as exc:
    raise ImportError(
        "Install the Anthropic SDK first: pip install -U anthropic"
    ) from exc

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. "
        "Set it in your environment before running API cells."
    )

client = Anthropic()

# Configurable because model availability can change.
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5")
MAX_ITERATIONS = 8

TOOLS = [
    {
        "name": "calculator",
        "description": (
            "Evaluate a basic arithmetic expression containing numbers, "
            "parentheses, and +, -, *, /, //, %, or **. "
            "Do not use variables, function calls, imports, or other Python syntax."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A basic arithmetic expression, e.g. '(12 * 3) + 5'."
                }
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_weather",
        "description": (
            "Return deterministic demo weather data for a city. "
            "This is a learning stub, not a live weather service. "
            "Input must be a city name such as Lahore, London, or New York."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City whose demo weather should be retrieved."
                }
            },
            "required": ["city"],
            "additionalProperties": False,
        },
    },
]

print(json.dumps(TOOLS, indent=2))

In [ ]:
_ALLOWED_BINARY_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.FloorDiv: op.floordiv,
    ast.Mod: op.mod,
    ast.Pow: op.pow,
}

def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)

    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
        value = _safe_eval(node.operand)
        return +value if isinstance(node.op, ast.UAdd) else -value

    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINARY_OPS:
        left = _safe_eval(node.left)
        right = _safe_eval(node.right)

        if isinstance(node.op, ast.Pow) and abs(right) > 10:
            raise ValueError("Exponent is too large for this demo calculator.")

        return _ALLOWED_BINARY_OPS[type(node.op)](left, right)

    raise ValueError("Unsupported expression. Use basic arithmetic only.")

def calculator(expression: str) -> Dict[str, Any]:
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree)
        return {"ok": True, "expression": expression, "result": result}
    except Exception as exc:
        return {"ok": False, "error": str(exc)}

WEATHER_DATA = {
    "lahore": {"temperature_c": 31, "condition": "Partly cloudy"},
    "london": {"temperature_c": 18, "condition": "Cloudy"},
    "new york": {"temperature_c": 24, "condition": "Sunny"},
    "karachi": {"temperature_c": 29, "condition": "Clear"},
}

def get_weather(city: str) -> Dict[str, Any]:
    key = city.strip().lower()
    if key not in WEATHER_DATA:
        return {
            "ok": False,
            "error": (
                f"No demo weather data for '{city}'. "
                "Try Lahore, London, New York, or Karachi."
            ),
        }

    return {"ok": True, "city": city, **WEATHER_DATA[key]}

TOOL_FUNCTIONS = {
    "calculator": calculator,
    "get_weather": get_weather,
}

def execute_tool(tool_name: str, tool_input: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name not in TOOL_FUNCTIONS:
        return {"ok": False, "error": f"Unknown tool requested: {tool_name}"}

    try:
        return TOOL_FUNCTIONS[tool_name](**tool_input)
    except TypeError as exc:
        return {"ok": False, "error": f"Invalid arguments for {tool_name}: {exc}"}
    except Exception as exc:
        return {"ok": False, "error": f"{tool_name} failed: {exc}"}

### Single-request tool call

This demonstrates:

`user request → model chooses tool → Python executes tool → tool_result returned`

It is intentionally separate from the full agent loop.

In [ ]:
single_request_messages = [
    {
        "role": "user",
        "content": "Use the calculator tool to calculate (18 * 7) + 4."
    }
]

response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    tools=TOOLS,
    messages=single_request_messages,
)

print("stop_reason:", response.stop_reason)

for block in response.content:
    if block.type == "tool_use":
        print("MODEL TOOL CHOICE:", block.name)
        print("TOOL INPUT:", block.input)

        result = execute_tool(block.name, block.input)
        print("TOOL RESULT:", result)

        single_request_messages.append(
            {"role": "assistant", "content": response.content}
        )
        single_request_messages.append(
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result),
                    }
                ],
            }
        )

        print("\nTool result block appended successfully.")
        break
else:
    print("The model did not request a tool on this run.")

## Task 3 — Build a Minimal Agent Loop

The agent below owns the message history, API call, tool detection, tool execution, tool-result messages, iteration counting, final-answer detection, error handling, and logging.

The loop uses a `for` range as the bounded equivalent of a `while` loop: the control flow is still repeated until a final answer or the maximum iteration limit is reached.

In [ ]:
def extract_text(response) -> str:
    """Collect visible text blocks from an Anthropic response."""
    parts = []
    for block in response.content:
        if block.type == "text":
            parts.append(block.text)
    return "\n".join(parts).strip()


def run_agent(user_request: str, max_iterations: int = MAX_ITERATIONS) -> str:
    """
    Minimal raw-Python agent loop.

    Flow:
        user -> model -> tool_use -> Python tool -> tool_result -> model -> ...
        until the model returns no tool_use or max_iterations is reached.
    """
    messages = [{"role": "user", "content": user_request}]

    for iteration in range(1, max_iterations + 1):
        print(f"\n{'=' * 70}")
        print(f"ITERATION {iteration}")
        print(f"{'=' * 70}")

        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages,
        )

        print("STOP REASON:", response.stop_reason)

        messages.append({
            "role": "assistant",
            "content": response.content
        })

        tool_uses = [
            block for block in response.content
            if block.type == "tool_use"
        ]

        if not tool_uses:
            final_text = extract_text(response)
            print("\nFINAL ANSWER:")
            print(final_text)
            return final_text

        tool_results = []

        for tool_use in tool_uses:
            print(f"\nACT → tool: {tool_use.name}")
            print("arguments:", tool_use.input)

            result = execute_tool(tool_use.name, tool_use.input)

            print("OBSERVE →", result)

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": json.dumps(result),
                **({"is_error": True} if not result.get("ok", True) else {}),
            })

        messages.append({
            "role": "user",
            "content": tool_results
        })

    raise RuntimeError(
        f"Agent stopped after max_iterations={max_iterations}. "
        "Possible cause: repeated tool calls or a task that never reaches completion."
    )

### Multi-step test: two cities

This task requires the model to retrieve weather for **two different cities** and then compare the observations.

In [ ]:
result = run_agent(
    "Look up the weather in Lahore and London using the weather tool. "
    "Then tell me which city is warmer and by how many degrees Celsius."
)

print("\nReturned result:")
print(result)

## Task 4 — Memory & State Handling

### Conversation memory

**Conversation memory** is the message history sent to the model: user messages, assistant responses, and tool results. In this notebook, `messages` is the conversation memory.

### Working memory

**Working memory** is structured state tracked by the program while solving the current task. Examples include the iteration counter, tool observations, cached intermediate values, IDs, or completion flags.

Working memory does not have to be sent to the model verbatim.

### Debugging habit

The loop logs each iteration, model stop reason, selected tool, arguments, observation, and final answer. In production, replace raw `print()` calls with structured logging and redact secrets or sensitive data.

## Task 5 — Failure Modes & Guardrails

We deliberately break the weather tool by requesting a city that is not in the stub.

In [ ]:
failure_test = run_agent(
    "Look up the weather in Atlantis. If the tool cannot find it, "
    "explain the limitation instead of inventing a temperature."
)

print("\nFailure-test result:")
print(failure_test)

### Observed failure behavior

For an unsupported city, the local tool returns a structured error instead of raising an unhandled exception. The error is sent back as a `tool_result` with `is_error=True`. The model can then recover by acknowledging that the requested data is unavailable rather than fabricating a temperature.

### Failure modes and mitigations

| Failure mode | What can happen | Mitigation |
|---|---|---|
| Infinite/repeating loop | Model repeatedly calls tools without finishing | `max_iterations` plus loop telemetry |
| Wrong tool arguments | Missing/invalid fields cause tool failures | Strict JSON schemas + server-side validation |
| Hallucinated/unknown tool | Model names a tool the application does not expose | Allowlisted dispatcher returns a structured error |
| Silent tool errors | A failed API/database call looks like valid data | Structured `{ok, error}` results + `is_error=True` |
| Unsafe tool execution | Model input reaches arbitrary code | Allowlist tools and validate inputs; avoid raw `eval()` |
| Stale/unavailable data | External data is incomplete or unavailable | Include status/source/timestamp and state limitations |

### Why frameworks exist

Frameworks such as LangChain, LangGraph, and CrewAI exist because the raw loop becomes repetitive and difficult to operate as systems grow. They provide abstractions for tool registration, state, retries, routing, persistence, tracing, human approval, multi-agent coordination, and graph/workflow control. The important lesson is that these frameworks are conveniences around primitives we can now see clearly: messages, tools, state, control flow, and guardrails.

## Final checklist

- [x] Agent vs. chatbot vs. workflow explained
- [x] Agentic characteristics explained
- [x] ReAct diagram and pseudocode included
- [x] Overkill judgment included
- [x] Two JSON tool schemas defined
- [x] Tool descriptions explained
- [x] Single tool-call round trip implemented
- [x] Raw Python loop implemented with a maximum-iteration safeguard
- [x] Multi-step task requiring 2+ tool calls included
- [x] Conversation memory vs. working memory explained
- [x] Logging included
- [x] Deliberate tool failure tested
- [x] Six failure modes with mitigations documented
- [x] Framework rationale included
- [x] API key is read from an environment variable rather than hard-coded